# Character-Level Dinosaur Name Generator

This notebook demonstrates how to build and train character-level language models using PyTorch to generate realistic-looking dinosaur names. We will implement both a custom **Recurrent Neural Network (RNN)** and a **Long Short-Term Memory (LSTM)** network from scratch.

In [170]:
import torch
import torch.nn as nn
import random
import torch.nn.functional as F

## 1. Setup and Hardware Acceleration

Here we detect if a GPU (`cuda`) is available to speed up training, and fallback to CPU if not.

In [171]:
device="cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

## 2. Data Loading & Text Preprocessing

We read our raw dinosaur dataset from `dinos.txt`, convert it to lowercase, build a set of unique characters, and format each dinosaur name to end with a dedicated stop token (`.`).

In [172]:
data=open('dinos.txt',"r").read()
data=data.lower()
chars=set(data)


In [173]:
chars

{'\n',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z'}

In [174]:
chars.remove('\n')
chars

{'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z'}

In [175]:
names=[name.strip()+'.'for name in data.splitlines()]

In [176]:
names[5]

'abrosaurus.'

In [177]:
n_chars,n_names=len(chars),len(names)
print(f"Number of names :{n_names} & number of characters {n_chars}")

Number of names :1536 & number of characters 26


In [178]:
chars=sorted(list(chars))
chars_dict={}
for index,char in enumerate(chars):
  chars_dict[char]=index
chars_dict['.']=26

In [179]:
chars_dict

{'a': 0,
 'b': 1,
 'c': 2,
 'd': 3,
 'e': 4,
 'f': 5,
 'g': 6,
 'h': 7,
 'i': 8,
 'j': 9,
 'k': 10,
 'l': 11,
 'm': 12,
 'n': 13,
 'o': 14,
 'p': 15,
 'q': 16,
 'r': 17,
 's': 18,
 't': 19,
 'u': 20,
 'v': 21,
 'w': 22,
 'x': 23,
 'y': 24,
 'z': 25,
 '.': 26}

In [180]:
index_to_char={}
for char,index in chars_dict.items():
  index_to_char[index]=char
index_to_char

{0: 'a',
 1: 'b',
 2: 'c',
 3: 'd',
 4: 'e',
 5: 'f',
 6: 'g',
 7: 'h',
 8: 'i',
 9: 'j',
 10: 'k',
 11: 'l',
 12: 'm',
 13: 'n',
 14: 'o',
 15: 'p',
 16: 'q',
 17: 'r',
 18: 's',
 19: 't',
 20: 'u',
 21: 'v',
 22: 'w',
 23: 'x',
 24: 'y',
 25: 'z',
 26: '.'}

## 3. Data Encoding & Batching

To train our models, we need to map characters to integers and group the sequences into batches. We define helper functions to encode names and dynamically pad batches to match the longest sequence in each batch.

In [181]:
START = 26
def encode(name):
    ids=[chars_dict[char]for char in name]
    y=ids
    x=[START]+ids[:-1]
    return x,y

In [182]:
def get_batches(names, batch_size):
   random.shuffle(names)
   for i in range(0,len(names),batch_size):
      batch=names[i:i+batch_size]
      L=max([len(name)for name in batch])
      X=torch.full((len(batch),L),START)
      Y=torch.full((len(batch),L),-100)
      for j,n in enumerate(batch):
        x,y=encode(n)
        X[j,:len(x)]=torch.tensor(x)
        Y[j,:len(y)]=torch.tensor(y)
      yield X.to(device),Y.to(device)



## 4. Custom RNN Model Implementation

We implement a standard character-level Recurrent Neural Network (RNN) using PyTorch linear layers and a tanh activation.

In [213]:
class RNN(nn.Module):
  def __init__(self,input_size=27,out_size=27,hidden_state=100):
    super().__init__()
    self.hidden_state=hidden_state
    self.Wa=nn.Linear(input_size+hidden_state,hidden_state)
    self.Wya=nn.Linear(hidden_state,out_size)

  def forward(self,x):
    outs=[]
    B,T =x.shape
    a=torch.zeros((B,self.hidden_state),dtype=torch.float32,device=device)
    for i in range(T):
      xt=F.one_hot(x[:,i],27).float().to(device)
      combined = torch.cat((xt, a), dim=1)
      a=torch.tanh(self.Wa(combined))
      yt=self.Wya(a)

      outs.append(yt)
    return torch.stack(outs,dim=1)

## 5. RNN Training

We initialize the RNN model, set up the Adam optimizer, and define a standard training loop with gradient clipping.

In [184]:
rnn=RNN()
optim=torch.optim.Adam(rnn.parameters(),lr=0.01)
criterion=nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(
    optim,
    step_size=500,
    gamma=0.5
)


In [217]:
def train(model,names,optim,criterion,n_epochs):
  model.to(device)
  model.train()
  for epoch in range(n_epochs):
    total=0
    n=0
    for X,Y in get_batches(names,128*2):
      optim.zero_grad()
      y_preds=model(X)
      Loss=criterion(y_preds.view(-1,27),Y.view(-1))
      Loss.backward()
      nn.utils.clip_grad_norm_(model.parameters(), 5)
      optim.step()
      total+=Loss.item()
      n+=1
    if (epoch+1)%50 ==0:
      print(f"epoch {epoch+1}: loss {total/n:.4f}")







In [186]:
train(rnn,names,optim,criterion,2000)

epoch 50: loss 1.2932
epoch 100: loss 1.0333
epoch 150: loss 0.9299
epoch 200: loss 0.8918
epoch 250: loss 0.8766
epoch 300: loss 0.8452
epoch 350: loss 0.8657
epoch 400: loss 0.8085
epoch 450: loss 0.8217
epoch 500: loss 0.8077
epoch 550: loss 0.6926
epoch 600: loss 0.7224
epoch 650: loss 0.6814
epoch 700: loss 0.7258
epoch 750: loss 0.6865
epoch 800: loss 0.7837
epoch 850: loss 0.6645
epoch 900: loss 0.8080
epoch 950: loss 0.6650
epoch 1000: loss 0.9299
epoch 1050: loss 0.6475
epoch 1100: loss 0.6356
epoch 1150: loss 0.6940
epoch 1200: loss 0.6525
epoch 1250: loss 0.6290
epoch 1300: loss 0.6693
epoch 1350: loss 0.6256
epoch 1400: loss 0.6275
epoch 1450: loss 0.7207
epoch 1500: loss 0.6254
epoch 1550: loss 0.6164
epoch 1600: loss 0.6132
epoch 1650: loss 0.6118
epoch 1700: loss 0.6136
epoch 1750: loss 0.6100
epoch 1800: loss 0.7064
epoch 1850: loss 0.6127
epoch 1900: loss 0.6073
epoch 1950: loss 0.6152
epoch 2000: loss 0.6064


## 6. Inference / Sampling

To generate new names, we start with a start token and iteratively sample the next character based on the model's output probability distribution.

In [211]:

def sample(model, index_to_char, max_length=20, temperature=0.8):

    model.eval()

    with torch.no_grad():
        x=torch.tensor([[START]],dtype=torch.int32,device=device)
        name=""
        for _ in range(max_length):
            logits=model(x)
            logits=logits[:,-1,:]/temperature
            probs=torch.softmax(logits,dim=-1)
            idx = torch.multinomial(probs, 1).item()
            if idx == START:
                break
            name += index_to_char[idx]
            next_char = torch.tensor([[idx]], dtype=torch.long, device=device)
            x = torch.cat([x, next_char], dim=1)
    return name


In [210]:
for i in range(20):
  print(sample(rnn, index_to_char,  max_length=30, temperature=1.0))

heconthicoptir
gigantosaurus
guaniagisaurus
herbstosaurus
pekinosaurus
riojasuchus
chaoyangsaurus
pieosaurus
skeleton
camposaurus
avalonianus
europelta
hexinlusaurus
dahuenatosaurus
richorhan
tatankacephalus
dracoraptor
zhuchengtitan
trachodon
spinostropheus


## 7. Custom LSTM Model Implementation

LSTMs introduce cell states and gating mechanisms (forget gate, input gate, output gate) to capture long-term dependencies much better than vanilla RNNs.

In [228]:
class LSTM(nn.Module):
  def __init__(self,input_size=27,out_size=27,hidden_state=100):
    super().__init__()
    self.hidden_state=hidden_state
    self.Wc=nn.Linear(input_size+hidden_state,hidden_state)
    self.Gu=nn.Linear(input_size+hidden_state,hidden_state)
    self.Gf=nn.Linear(input_size+hidden_state,hidden_state)
    self.Go=nn.Linear(input_size+hidden_state,hidden_state)
    self.Wya=nn.Linear(hidden_state,out_size)

  def forward(self,x):
    outs=[]
    B,T =x.shape
    ct=torch.zeros((B,self.hidden_state),dtype=torch.float32,device=device)
    at = torch.zeros((B,self.hidden_state), dtype=torch.float32, device=device)
    for i in range(T):
      xt=F.one_hot(x[:,i].long(),27).float().to(device)
      combined = torch.cat((xt, at), dim=1)
      ct_candicate=torch.tanh(self.Wc(combined))
      Gu=torch.sigmoid(self.Gu(combined))
      Gf=torch.sigmoid(self.Gf(combined))
      Go=torch.sigmoid(self.Go(combined))
      ct=Gu*ct_candicate+Gf*ct
      at=Go*torch.tanh(ct)
      yt=self.Wya(at)
      outs.append(yt)
    return torch.stack(outs,dim=1)

## 8. LSTM Training & Evaluation

We initialize, train, and sample from our custom LSTM to compare its generated names with the RNN.

In [229]:
lstm=LSTM()
optim=torch.optim.Adam(lstm.parameters(),lr=0.005)
criterion=nn.CrossEntropyLoss()


In [230]:
train(lstm,names,optim,criterion,500)

epoch 50: loss 1.4698
epoch 100: loss 0.9790
epoch 150: loss 0.7473
epoch 200: loss 0.6632
epoch 250: loss 0.6243
epoch 300: loss 0.6079
epoch 350: loss 0.5966
epoch 400: loss 0.5906
epoch 450: loss 0.5872
epoch 500: loss 0.5856


In [231]:
for i in range(20):
  print(sample(lstm, index_to_char,  max_length=30, temperature=1.0))

roceratops
palaeoctonus
dillodon
khetranisaurus
pamparaptor
condorraptor
graciliceratops
ojoraptorsaurus
metriacanthosaurus
palaeosaurus
caudocoelus
cryptodraco
macrurosaurus
chenanisaurus
stereocephalus
xiaosaurus
tawa
dongyangopelta
sphaerotholus
kentrosaurus
